In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("/content/twitter_sentiment_data.csv")

In [ ]:
df.head()

,sentiment,message,tweetid
0,-1,@tiniebeany climate change is an interesting h...,792927353886371840
1,1,RT @NatGeoChannel: Watch #BeforeTheFlood right...,793124211518832641
2,1,Fabulous! Leonardo #DiCaprio's film on #climat...,793124402388832256
3,1,RT @Mick_Fanning: Just watched this amazing do...,793124635873275904
4,2,"RT @cnalive: Pranita Biswasi, a Lutheran from ...",793125156185137153


In [ ]:
import pandas as pd

# Check if df is not a DataFrame or is None, and reload if necessary
if 'df' not in locals() or not isinstance(df, pd.DataFrame):
    df = pd.read_csv("/content/twitter_sentiment_data.csv")

df.drop('tweetid', axis=1, inplace=True, errors='ignore')

In [ ]:
df.head()

,sentiment,message
0,-1,@tiniebeany climate change is an interesting h...
1,1,RT @NatGeoChannel: Watch #BeforeTheFlood right...
2,1,Fabulous! Leonardo #DiCaprio's film on #climat...
3,1,RT @Mick_Fanning: Just watched this amazing do...
4,2,"RT @cnalive: Pranita Biswasi, a Lutheran from ..."


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
1,22962
2,9276
0,7715
-1,3990


In [ ]:
df = df[df['sentiment'] != 2]
display(df['sentiment'].value_counts())

,count
sentiment,
1,22962
0,7715
-1,3990


In [ ]:
# Map sentiment values to descriptive labels
sentiment_mapping = {
    1: 'Favor',
    0: 'None',
    -1: 'Against'
}
df['sentiment'] = df['sentiment'].replace(sentiment_mapping)

# Display the updated sentiment counts
display(df['sentiment'].value_counts())

,count
sentiment,
Favor,22962
None,7715
Against,3990


In [ ]:
import pandas as pd
import re
import html
import unicodedata

In [ ]:
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (34667, 2)


,sentiment,message
0,Against,@tiniebeany climate change is an interesting h...
1,Favor,RT @NatGeoChannel: Watch #BeforeTheFlood right...
2,Favor,Fabulous! Leonardo #DiCaprio's film on #climat...
3,Favor,RT @Mick_Fanning: Just watched this amazing do...
5,None,Unamshow awache kujinga na iko global warming ...


In [ ]:
df = df.dropna(subset=['message'])

print("After removing null values:", df.shape)

After removing null values: (34667, 2)


In [ ]:
df = df.drop_duplicates(subset=['message'])

print("After removing duplicates:", df.shape)

After removing duplicates: (31962, 2)


In [ ]:
def clean_tweet(text):

    # Convert to string
    text = str(text)

    # ---------------------------------------------------
    # Decode HTML entities
    # Example: &amp; -> &
    # ---------------------------------------------------
    text = html.unescape(text)

    # ---------------------------------------------------
    # Normalize Unicode
    # ---------------------------------------------------
    text = unicodedata.normalize("NFKC", text)

    # ---------------------------------------------------
    # Remove RT at beginning
    # ---------------------------------------------------
    text = re.sub(r'^RT\s+', '', text)

    # ---------------------------------------------------
    # Remove Twitter shortened URLs
    # Example:
    # https://t.co/7rV6Br
    # ---------------------------------------------------
    text = re.sub(r'https?://t\.co/\S+', '', text)

    # ---------------------------------------------------
    # Remove all URLs
    # ---------------------------------------------------
    text = re.sub(r'https?://\S+|http?://\S+|www\.\S+', '', text)

    # ---------------------------------------------------
    # Remove HTML tags
    # ---------------------------------------------------
    text = re.sub(r'<.*?>', '', text)

    # ---------------------------------------------------
    # Remove user mentions
    # Example:
    # @elonmusk
    # ---------------------------------------------------
    text = re.sub(r'@\w+', '', text)

    # ---------------------------------------------------
    # Remove hashtag symbol only
    # Example:
    # #Climate -> Climate
    # ---------------------------------------------------
    text = re.sub(r'#(\w+)', r'\1', text)

    # ---------------------------------------------------
    # Remove emojis
    # ---------------------------------------------------
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub('', text)

    # ---------------------------------------------------
    # Fix common encoding problems
    # ---------------------------------------------------
    replacements = {
        "â€™": "'",
        "â€œ": '"',
        "â€": '"',
        "â€˜": "'",
        "â€“": "-",
        "â€”": "-",
        "…": "...",
        "Â": "",
        "\xa0": " "
    }

    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)

    # ---------------------------------------------------
    # Remove repeated punctuation
    # Example:
    # !!!! -> !
    # ????? -> ?
    # ..... -> .
    # ---------------------------------------------------
    text = re.sub(r'([!?.,])\1+', r'\1', text)

    # ---------------------------------------------------
    # Remove extra brackets left after URL removal
    # ---------------------------------------------------
    text = re.sub(r'\(\s*\)', '', text)
    text = re.sub(r'\[\s*\]', '', text)
    text = re.sub(r'\{\s*\}', '', text)

    # ---------------------------------------------------
    # Normalize whitespace
    # ---------------------------------------------------
    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

In [ ]:
df['clean_text'] = df['message'].apply(clean_tweet)

In [ ]:
df = df[df['clean_text'].str.strip() != ""]

print("After removing empty tweets:", df.shape)

After removing empty tweets: (31947, 3)


In [ ]:
df = df[df['clean_text'].str.split().str.len() >= 3]

print("After removing very short tweets:", df.shape)

After removing very short tweets: (31837, 3)


In [ ]:
for i in range(10):

    print("="*100)

    print("Original Tweet:\n")
    print(df.iloc[i]['message'])

    print("\nCleaned Tweet:\n")
    print(df.iloc[i]['clean_text'])

Original Tweet:

@tiniebeany climate change is an interesting hustle as it was global warming but the planet stopped warming for 15 yes while the suv boom

Cleaned Tweet:

climate change is an interesting hustle as it was global warming but the planet stopped warming for 15 yes while the suv boom
Original Tweet:

RT @NatGeoChannel: Watch #BeforeTheFlood right here, as @LeoDiCaprio travels the world to tackle climate change https://t.co/LkDehj3tNn httÃ¢â‚¬Â¦

Cleaned Tweet:

: Watch BeforeTheFlood right here, as travels the world to tackle climate change httÃ¢â‚¬¦
Original Tweet:

Fabulous! Leonardo #DiCaprio's film on #climate change is brilliant!!! Do watch. https://t.co/7rV6BrmxjW via @youtube

Cleaned Tweet:

Fabulous! Leonardo DiCaprio's film on climate change is brilliant! Do watch. via
Original Tweet:

RT @Mick_Fanning: Just watched this amazing documentary by leonardodicaprio on climate change. We all think thisÃ¢â‚¬Â¦ https://t.co/kNSTE8K8im

Cleaned Tweet:

: Just watched this

In [ ]:
df.head()

,sentiment,message,clean_text
0,Against,@tiniebeany climate change is an interesting h...,climate change is an interesting hustle as it ...
1,Favor,RT @NatGeoChannel: Watch #BeforeTheFlood right...,": Watch BeforeTheFlood right here, as travels ..."
2,Favor,Fabulous! Leonardo #DiCaprio's film on #climat...,Fabulous! Leonardo DiCaprio's film on climate ...
3,Favor,RT @Mick_Fanning: Just watched this amazing do...,: Just watched this amazing documentary by leo...
5,None,Unamshow awache kujinga na iko global warming ...,Unamshow awache kujinga na iko global warming


In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
Favor,20566
None,7426
Against,3845


In [ ]:
favor = df[df["sentiment"] == "Favor"]
none = df[df["sentiment"] == "None"]
against = df[df["sentiment"] == "Against"]

In [ ]:
favor = favor.sample(n=3845, random_state=42)

none = none.sample(n=3845, random_state=42)

# Against already has 3845 samples
against = against.sample(frac=1, random_state=42)

In [ ]:
balanced_df = pd.concat(
    [favor, none, against],
    ignore_index=True
)

In [ ]:
balanced_df = balanced_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [ ]:
print(balanced_df["sentiment"].value_counts())

sentiment
None       3845
Favor      3845
Against    3845
Name: count, dtype: int64


In [ ]:
balanced_df.to_csv(
    "twitter_dataset.csv",
    index=False
)

print("Balanced dataset saved successfully!")

Balanced dataset saved successfully!
